In [ ]:
import os
from pathlib import Path
import sys

if Path.cwd().name == "notebooks":
  os.chdir(Path.cwd().parent.parent)

print("Intérprete:", sys.executable)
print("Carpeta de trabajo:", Path.cwd())

for lib in ["numpy", "pandas", "matplotlib", "sklearn"]:
  try:
    print(f" {lib:12}", __import__(lib).__version__)
  except ImportError:
    print(f" {lib:12} NO DISPONIBLE en este kernel")

# Fase 1: Definición del Problema, Entorno Reproducible y Validación del Dataset
**Curso:** MCDI500 — Programación para la Ciencia de Datos (Grupo 3)  
**Integrantes:** Madeleine Aguilera, Victoria Álvarez, Ignacio Peña y Raúl Hormazábal

---

## 1. Situación Problematizadora y Pregunta Analizable
El monitoreo meteorológico en Chile mediante estaciones automáticas de la Dirección Meteorológica de Chile (DMC) y el Explorador Climático CR2 permite estudiar eventos de precipitación y anomalías térmicas a través de las cuatro macrozonas del país (Norte, Centro, Sur y Austral). Sin embargo, los registros operacionales diarios presentan **vacíos reales de captura en sensores (entre 4,5% y 14,2%)**, **valores extremos (outliers)** durante sistemas frontales u olas de calor, e **inconsistencias de formato en etiquetas categóricas**.

- **Pregunta analizable del proyecto:** *¿Qué variables meteorológicas y termodinámicas diarias se asocian con la ocurrencia de precipitaciones y con los niveles de alerta térmica entre las distintas macrozonas de Chile durante el período 2021–2025?*
- **Objetivo General:** Implementar un flujo de trabajo reproducible y trazable en Python para explorar, limpiar, transformar y analizar los registros meteorológicos diarios de estaciones chilenas (2021–2025).
- **Objetivos Específicos (Fases 1 y 2):**
  1. Configurar y verificar un entorno computacional aislado (`.venv`), conectado al kernel del proyecto y versionado en Git/GitHub con higiene de notebooks (`nbstripout`).
  2. Validar que el conjunto de datos original (`F1/data/raw/clima_estaciones_chile_raw.csv`) cumple los requisitos multivariados y de volumen del curso sin alterar el archivo crudo.
  3. Diseñar en la Fase 2 un pipeline modular de limpieza, imputación por grupo (`macrozona`), codificación según rol analítico y escalamiento robusto (`RobustScaler`).

In [ ]:
# Celda 3: Carga controlada del dataset original, diccionario de variables y validación Fase 1
import pandas as pd
from pathlib import Path
from IPython.display import display

RUTA_RAW = Path("F1/data/raw/clima_estaciones_chile_raw.csv")

# 1. Carga con manejo de excepciones (buena práctica de software científico)
try:
    if not RUTA_RAW.exists():
        raise FileNotFoundError(f"No se encontró el archivo en la ruta relativa: {RUTA_RAW}")
    df_raw = pd.read_csv(RUTA_RAW, encoding="utf-8")
    print(f"[OK] Dataset cargado correctamente desde: {RUTA_RAW}")
    print(f"Dimensiones originales: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas")
except Exception as e:
    print(f"[ERROR] Fallo al cargar el dataset: {e}")
    raise

# 2. Diccionario de las 16 variables definido por el Grupo 3 (cubre los 8 roles del curso)
DICCIONARIO_VARIABLES = {
    "id_registro":                   ("identificador",     "Código único de estación y fecha (DMC-XXXXXX-YYYYMMDD)"),
    "fecha_observacion":             ("fecha",             "Fecha diaria de medición (YYYY-MM-DD, período 2021-2025)"),
    "codigo_estacion_comuna":        ("alta cardinalidad", "Código DMC y nombre de la estación/comuna (30 estaciones)"),
    "macrozona":                     ("nominal",           "Macrozona climática de Chile (Norte, Centro, Sur, Austral)"),
    "temp_max_c":                    ("continua",          "Temperatura máxima diaria del aire (°C)"),
    "temp_min_c":                    ("continua",          "Temperatura mínima diaria del aire (°C)"),
    "humedad_relativa_pct":          ("continua",          "Humedad relativa media diaria (%)"),
    "presion_hpa":                   ("continua",          "Presión atmosférica media a nivel de estación (hPa)"),
    "velocidad_viento_kmh":          ("continua",          "Velocidad máxima diaria del viento (km/h)"),
    "precipitacion_mm":              ("continua",          "Precipitación acumulada en 24 horas (mm)"),
    "nubosidad_octas":               ("discreta",          "Cobertura nubosa diaria medida en octavos de cielo (0 a 8 octas)"),
    "dias_consecutivos_secos":       ("discreta",          "Recuento entero de días seguidos sin precipitación significativa"),
    "direccion_viento_predominante": ("nominal",           "Punto cardinal de dirección predominante del viento (N, S, SW, etc.)"),
    "ocurrencia_lluvia":             ("binaria",           "Indicador dicotómico de día lluvioso (Si / No)"),
    "nivel_alerta_termica":          ("ordinal",           "Escala ordenada de severidad térmica: Normal < Alerta Temprana < Alerta < Alarma"),
    "observacion_sinoptica":         ("texto libre",       "Reporte descriptivo de la condición sinóptica diaria"),
}

# 3. Construcción del perfil técnico de las variables
perfil_f1 = pd.DataFrame([
    {
        "variable": col,
        "dtype_pandas": str(df_raw[col].dtype),
        "rol_analitico": DICCIONARIO_VARIABLES[col][0],
        "valores_unicos": int(df_raw[col].nunique()),
        "nulos_n": int(df_raw[col].isna().sum()),
        "nulos_pct": round(df_raw[col].isna().mean() * 100, 1),
        "descripcion": DICCIONARIO_VARIABLES[col][1],
    }
    for col in df_raw.columns
])

display(perfil_f1)

# 4. Verificación formal de los umbrales mínimos del curso mediante assert
roles_presentes = set(perfil_f1["rol_analitico"]) - {"identificador"}
max_pct_nulos = perfil_f1["nulos_pct"].max()

assert df_raw.shape[0] >= 2000, "El dataset debe tener al menos 2.000 filas"
assert df_raw.shape[1] >= 12, "El dataset debe tener al menos 12 columnas"
assert len(roles_presentes) >= 7, "Debe cubrir múltiples roles analíticos"
assert 1.0 <= max_pct_nulos <= 60.0, "El porcentaje máximo de nulos debe estar entre 1% y 60%"

print("\nResumen de roles analíticos presentes:")
print(perfil_f1["rol_analitico"].value_counts().to_string())
print(f"\n[OK] Todos los requisitos verificados. Máximo de nulos: {max_pct_nulos}% en velocidad_viento_kmh.")

# 5. Guardar reporte Markdown de validación en F1/docs/informe_dataset.md
ruta_informe = Path("F1/docs/informe_dataset.md")
lineas_md = [
    "# Informe de Validación del Dataset (Fase 1) — Grupo 3 MCDI500",
    f"- **Archivo evaluado:** `{RUTA_RAW}`",
    f"- **Dimensiones:** {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas",
    f"- **Filas duplicadas exactas:** {int(df_raw.duplicated().sum())}",
    f"- **Rango de valores faltantes en columnas con nulos:** 4,5% a {max_pct_nulos}%",
    "",
    "## Diccionario y Perfil de Variables",
    "| " + " | ".join(perfil_f1.columns) + " |",
    "| " + " | ".join(["---"] * len(perfil_f1.columns)) + " |",
]
for fila in perfil_f1.values:
    lineas_md.append("| " + " | ".join(str(v) for v in fila) + " |")

ruta_informe.write_text("\n".join(lineas_md), encoding="utf-8")
print(f"[OK] Reporte exportado en: {ruta_informe}")